# Task MP-DATA-005: Feb-Mar 2026 Price Discrepancy & Train-Only Eligibility
**Repository:** `ankit-choubey/Mandi_Prediction`  
**Execution Environment:** Kaggle CPU Kernel (Internet Disabled)  
**Evaluated Dataset:** `theankitchoubey/mandi-prediction-audit-003b` (Version 1)  

### Objectives
1. **The 77 Price Differences Deep Dive:** Analyze price ratios, decimals, constant factor groupings, and raw Excel Price/Arrival unit headers.
2. **Cluster Scope Analysis:** Profile match rates by calendar week and commodity across all Excel rows from 2026-02-15 to 2026-04-07.
3. **Panel-Wide Anomaly Scan:** Scan all 462,350 panel rows for day-over-day price jumps, non-integers, 3x median outliers, and min/max inconsistencies across months and commodities.
4. **Train-Only Eligibility (Tiers A & B):** Re-derive series eligibility using strictly training window data (up to 2025-12-18) for Tier A (obs>=60, cov>=0.3) and Tier B (obs>=120, cov>=0.3), breaking out the six focus crops.
5. **Serving-Time Rule Evaluation:** Measure series recency at training cutoff and test-window serving viability.
6. **Series-Level Forward Availability:** Profile unweighted series distributions of T+7 and T+6..T+8 availability across tiers.
7. **Rolling-Origin Feasibility:** Evaluate 4 expanding-window test folds of 60 days each with a 7-date embargo ending 2026-09-30, identifying discrepancy overlap.
8. **Environment & Input Verification:** Record library versions, seeds (42), and SHA-256 hashes of all 30 input files.


In [ ]:
# Cell 1: Environment, Library Versions, Random Seeds, and Input Discovery
import os
import sys
import glob
import json
import hashlib
import time
import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

print("=== ENVIRONMENT & LIBRARY VERSIONS ===")
print(f"Python: {sys.version}")
print(f"Platform: {sys.platform}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"openpyxl: {openpyxl.__version__}")

# Random seed
np.random.seed(42)
print("Random seed initialized to 42.")

# Locate dataset root
KAGGLE_INPUT_DIR = None
search_candidates = [
    "/kaggle/input/mandi-prediction-audit-003b",
    "/kaggle/input/datasets/theankitchoubey/mandi-prediction-audit-003b",
    "/kaggle/input",
    "."
]
for candidate in search_candidates:
    if os.path.exists(candidate):
        if os.path.exists(os.path.join(candidate, "5 year dataset")):
            KAGGLE_INPUT_DIR = candidate
            break
        elif os.path.exists(os.path.join(candidate, "mandi-prediction-audit-003b", "5 year dataset")):
            KAGGLE_INPUT_DIR = os.path.join(candidate, "mandi-prediction-audit-003b")
            break

if KAGGLE_INPUT_DIR is None:
    matches = glob.glob("/kaggle/input/**/GHAZIABAD.csv", recursive=True) or glob.glob("**/GHAZIABAD.csv", recursive=True)
    if matches:
        KAGGLE_INPUT_DIR = str(Path(matches[0]).parent.parent)

print(f"Resolved Dataset Payload Root: {KAGGLE_INPUT_DIR}")
if KAGGLE_INPUT_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_INPUT_DIR)
src_path = os.path.join(KAGGLE_INPUT_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

def compute_sha256(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

print("\n=== INPUT FILES & SHA-256 MANIFEST ===")
input_file_records = []
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    for f in sorted(files):
        if f.endswith((".csv", ".json", ".py", ".xlsx", ".txt")):
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, KAGGLE_INPUT_DIR)
            sz = os.path.getsize(full_p)
            sha = compute_sha256(full_p)
            input_file_records.append({"file": rel_p, "size_bytes": sz, "sha256": sha})
            print(f"{sz:>10} bytes | {sha[:16]}... | {rel_p}")

print(f"Total verified input files: {len(input_file_records)}")


In [ ]:
# Cell 2: Part 1 - Deep Dive Analysis of the 77 Price Differences
from src.ingest_agmarknet_arrivals import load_arrivals_file

# 1. Load validated panel
val_csv_path = os.path.join(KAGGLE_INPUT_DIR, "data/processed/mandi_5year_validated.csv")
val_df = pd.read_csv(val_csv_path)
val_df["Arrival_Date"] = pd.to_datetime(val_df["Arrival_Date"]).dt.strftime("%Y-%m-%d")
val_df["Modal_Price_Clean"] = pd.to_numeric(val_df["Modal_Price"], errors="coerce")
print(f"Loaded validated panel: {len(val_df):,} rows")

# 2. Load Excel files with raw units
excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(os.path.join(KAGGLE_INPUT_DIR, "data/raw/arrivals/historical/raw_downloads/*.xlsx")))
if not excel_files:
    excel_files = sorted(glob.glob(f"{KAGGLE_INPUT_DIR}/**/raw_downloads/*.xlsx", recursive=True))

excel_dfs = []
for ef in excel_files:
    df_e = load_arrivals_file(ef)
    excel_dfs.append(df_e)
df_excel = pd.concat(excel_dfs, ignore_index=True)
df_excel["Modal_Price_Clean"] = pd.to_numeric(df_excel["Modal_Price"], errors="coerce")
df_excel["Arrival_Date"] = pd.to_datetime(df_excel["Arrival_Date"]).dt.strftime("%Y-%m-%d")

# Merge on natural key
nat_key_excel = ["District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date"]
nat_key_panel = ["District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date"]

merged_excel = pd.merge(
    df_excel,
    val_df,
    left_on=nat_key_excel,
    right_on=nat_key_panel,
    how="inner",
    suffixes=("_excel", "_panel")
)
merged_excel["price_diff"] = merged_excel["Modal_Price_Clean_excel"] - merged_excel["Modal_Price_Clean_panel"]
diff_77 = merged_excel[merged_excel["price_diff"] != 0].copy()
print(f"Verified non-zero price difference count: {len(diff_77)}")

# Compute ratios and decimals
diff_77["ratio"] = diff_77["Modal_Price_Clean_excel"] / diff_77["Modal_Price_Clean_panel"]
diff_77["excel_has_decimals"] = (diff_77["Modal_Price_Clean_excel"] % 1 != 0)
diff_77["panel_has_decimals"] = (diff_77["Modal_Price_Clean_panel"] % 1 != 0)

# Check constant factor candidate explanations (0.001, 0.01, 0.1, 10, 100, 1000)
factors = [0.001, 0.01, 0.1, 0.5, 2.0, 10.0, 100.0, 1000.0]
factor_matches = {}
for fac in factors:
    # Within 1% tolerance of constant factor
    match_mask = np.isclose(diff_77["ratio"], fac, rtol=0.01)
    factor_matches[str(fac)] = int(match_mask.sum())

print("\n=== RATIO & CONSTANT FACTOR ANALYSIS ===")
print("Constant Factor Matches (within 1% tolerance):", factor_matches)

# Ratio distribution
ratios = diff_77["ratio"]
ratio_dist = {
    "count": len(ratios),
    "min_ratio": round(float(ratios.min()), 4),
    "p10_ratio": round(float(np.percentile(ratios, 10)), 4),
    "p25_ratio": round(float(np.percentile(ratios, 25)), 4),
    "median_ratio": round(float(np.median(ratios)), 4),
    "p75_ratio": round(float(np.percentile(ratios, 75)), 4),
    "p90_ratio": round(float(np.percentile(ratios, 90)), 4),
    "max_ratio": round(float(ratios.max()), 4),
    "mean_ratio": round(float(ratios.mean()), 4)
}
print("Ratio Distribution:", ratio_dist)

# Unit analysis
print("\n=== UNIT COLUMN VALUES IN EXCEL FOR THE 77 ROWS ===")
print("Price Unit counts:")
if "Price_Unit" in diff_77.columns:
    print(diff_77["Price_Unit"].value_counts(dropna=False))
elif "Price Unit" in diff_77.columns:
    print(diff_77["Price Unit"].value_counts(dropna=False))

print("Arrival Unit counts:")
if "Arrival_Unit_Source" in diff_77.columns:
    print(diff_77["Arrival_Unit_Source"].value_counts(dropna=False))

# Decimals analysis
print("\nDecimals Analysis:")
print(f"Excel quotes with non-zero decimals: {diff_77['excel_has_decimals'].sum()} / {len(diff_77)}")
print(f"Panel quotes with non-zero decimals: {diff_77['panel_has_decimals'].sum()} / {len(diff_77)}")

# Export detailed 77 differences CSV
cols_to_export = [
    "District_Std", "Market", "Commodity_Std", "Variety", "Grade", "Arrival_Date",
    "Modal_Price_Clean_excel", "Modal_Price_Clean_panel", "price_diff", "ratio",
    "excel_has_decimals", "panel_has_decimals"
]
for col in ["Price_Unit", "Price Unit", "Arrival_Unit_Source", "Arrival Unit"]:
    if col in diff_77.columns and col not in cols_to_export:
        cols_to_export.append(col)

diff_77_out = diff_77[cols_to_export].copy()
diff_77_out.to_csv("diff_77_deep_analysis.csv", index=False)
print(f"Exported diff_77_deep_analysis.csv ({len(diff_77_out)} rows)")

diff_77_summary = {
    "total_diff_rows": len(diff_77),
    "ratio_distribution": ratio_dist,
    "constant_factor_matches": factor_matches,
    "excel_has_decimals_count": int(diff_77["excel_has_decimals"].sum()),
    "panel_has_decimals_count": int(diff_77["panel_has_decimals"].sum()),
    "records": diff_77_out.to_dict(orient="records")
}
with open("diff_77_deep_analysis.json", "w") as f:
    json.dump(diff_77_summary, f, indent=2)
print("Saved diff_77_deep_analysis.json")



In [ ]:
# Cell 3: Part 2 - Cluster Scope Analysis (2026-02-15 to 2026-04-07)
# Filter Excel rows in target cluster window
cluster_mask = (df_excel["Arrival_Date"] >= "2026-02-15") & (df_excel["Arrival_Date"] <= "2026-04-07")
excel_cluster = df_excel[cluster_mask].copy()
print(f"Total Excel rows in 2026-02-15..2026-04-07 window: {len(excel_cluster):,}")

# Match against validated panel
excel_cluster_merged = pd.merge(
    excel_cluster,
    val_df,
    left_on=nat_key_excel,
    right_on=nat_key_panel,
    how="left",
    suffixes=("_excel", "_panel")
)
excel_cluster_merged["is_panel_matched"] = excel_cluster_merged["Modal_Price_Clean_panel"].notna()
excel_cluster_merged["price_diff"] = excel_cluster_merged["Modal_Price_Clean_excel"] - excel_cluster_merged["Modal_Price_Clean_panel"]
excel_cluster_merged["is_exact_match"] = (excel_cluster_merged["price_diff"] == 0)

# Add week identifier (Monday of each week)
dt_col = pd.to_datetime(excel_cluster_merged["Arrival_Date"])
excel_cluster_merged["week_start"] = dt_col.apply(lambda d: (d - datetime.timedelta(days=d.weekday())).strftime("%Y-%m-%d"))

# Weekly match rates breakdown
weekly_grp = excel_cluster_merged.groupby("week_start").agg(
    total_excel_rows=("Modal_Price_Clean_excel", "count"),
    matched_panel_rows=("is_panel_matched", "sum"),
    exact_price_match_rows=("is_exact_match", "sum")
).reset_index()

weekly_grp["panel_match_pct"] = round(weekly_grp["matched_panel_rows"] / weekly_grp["total_excel_rows"] * 100, 2)
weekly_grp["exact_price_pct"] = round(weekly_grp["exact_price_match_rows"] / weekly_grp["matched_panel_rows"] * 100, 2)

print("\n=== WEEKLY MATCH RATES IN CLUSTER WINDOW ===")
print(weekly_grp.to_string(index=False))

# Commodity breakdown in cluster window
comm_grp = excel_cluster_merged.groupby("Commodity_Std").agg(
    total_excel_rows=("Modal_Price_Clean_excel", "count"),
    matched_panel_rows=("is_panel_matched", "sum"),
    exact_price_match_rows=("is_exact_match", "sum")
).reset_index()
comm_grp["panel_match_pct"] = round(comm_grp["matched_panel_rows"] / comm_grp["total_excel_rows"] * 100, 2)
comm_grp["exact_price_pct"] = round(comm_grp["exact_price_match_rows"] / comm_grp["matched_panel_rows"] * 100, 2)

print("\n=== COMMODITY MATCH RATES IN CLUSTER WINDOW ===")
print(comm_grp.to_string(index=False))

# Export weekly breakdown
weekly_grp.to_csv("feb_mar_cluster_weekly.csv", index=False)
print("Saved feb_mar_cluster_weekly.csv")

cluster_summary = {
    "cluster_window": "2026-02-15 to 2026-04-07",
    "total_excel_rows": len(excel_cluster),
    "matched_panel_rows": int(excel_cluster_merged["is_panel_matched"].sum()),
    "exact_match_rows": int(excel_cluster_merged["is_exact_match"].sum()),
    "overall_exact_match_pct": round(float(excel_cluster_merged["is_exact_match"].sum() / excel_cluster_merged["is_panel_matched"].sum() * 100), 2),
    "weekly_breakdown": weekly_grp.to_dict(orient="records"),
    "commodity_breakdown": comm_grp.to_dict(orient="records")
}
with open("feb_mar_cluster_summary.json", "w") as f:
    json.dump(cluster_summary, f, indent=2)
print("Saved feb_mar_cluster_summary.json")



In [ ]:
# Cell 4: Part 3 - Panel-Wide Anomaly Scan (No Deletions)
print("Executing panel-wide anomaly scan across all 462,350 validated rows...")

val_df_scan = val_df.copy()
val_df_scan["Arrival_Date_dt"] = pd.to_datetime(val_df_scan["Arrival_Date"])
val_df_scan.sort_values(by=["series_id", "Arrival_Date_dt"], inplace=True)

# 1. Day-over-day price jump flag: ratio outside [0.5, 2.0]
val_df_scan["prev_modal"] = val_df_scan.groupby("series_id")["Modal_Price_Clean"].shift(1)
val_df_scan["dod_ratio"] = val_df_scan["Modal_Price_Clean"] / val_df_scan["prev_modal"]
val_df_scan["flag_price_jump"] = val_df_scan["prev_modal"].notna() & ((val_df_scan["dod_ratio"] < 0.5) | (val_df_scan["dod_ratio"] > 2.0))

# 2. Non-integer prices flag
val_df_scan["flag_non_integer"] = (val_df_scan["Modal_Price_Clean"] % 1 != 0)

# 3. Outside 3x series median flag
series_medians = val_df_scan.groupby("series_id")["Modal_Price_Clean"].transform("median")
val_df_scan["series_median"] = series_medians
val_df_scan["flag_median_3x"] = (val_df_scan["Modal_Price_Clean"] > 3.0 * series_medians) | (val_df_scan["Modal_Price_Clean"] < (1.0 / 3.0) * series_medians)

# 4. Min/Max inconsistency flag
val_df_scan["Min_Price_Clean"] = pd.to_numeric(val_df_scan["Min_Price"], errors="coerce")
val_df_scan["Max_Price_Clean"] = pd.to_numeric(val_df_scan["Max_Price"], errors="coerce")
val_df_scan["flag_min_max_inconsistency"] = (
    (val_df_scan["Min_Price_Clean"] > val_df_scan["Max_Price_Clean"]) |
    (val_df_scan["Modal_Price_Clean"] < val_df_scan["Min_Price_Clean"]) |
    (val_df_scan["Modal_Price_Clean"] > val_df_scan["Max_Price_Clean"])
)

# Combined anomaly flag
val_df_scan["any_anomaly"] = (
    val_df_scan["flag_price_jump"] |
    val_df_scan["flag_non_integer"] |
    val_df_scan["flag_median_3x"] |
    val_df_scan["flag_min_max_inconsistency"]
)

total_rows = len(val_df_scan)
n_jump = int(val_df_scan["flag_price_jump"].sum())
n_non_int = int(val_df_scan["flag_non_integer"].sum())
n_med_3x = int(val_df_scan["flag_median_3x"].sum())
n_min_max = int(val_df_scan["flag_min_max_inconsistency"].sum())
n_any = int(val_df_scan["any_anomaly"].sum())

print(f"Total Rows Scanned: {total_rows:,}")
print(f"Flagged Price Jumps (DoD ratio <0.5 or >2.0): {n_jump:,} ({n_jump/total_rows*100:.2f}%)")
print(f"Flagged Non-Integer Prices: {n_non_int:,} ({n_non_int/total_rows*100:.2f}%)")
print(f"Flagged Outside 3x Median: {n_med_3x:,} ({n_med_3x/total_rows*100:.2f}%)")
print(f"Flagged Min/Max Inconsistency: {n_min_max:,} ({n_min_max/total_rows*100:.2f}%)")
print(f"Total Rows with Any Anomaly Flag: {n_any:,} ({n_any/total_rows*100:.2f}%)")

# Monthly anomaly distribution
val_df_scan["year_month"] = val_df_scan["Arrival_Date"].str[:7]
monthly_anomaly = val_df_scan.groupby("year_month").agg(
    total_obs=("Modal_Price_Clean", "count"),
    jump_count=("flag_price_jump", "sum"),
    non_int_count=("flag_non_integer", "sum"),
    med_3x_count=("flag_median_3x", "sum"),
    min_max_count=("flag_min_max_inconsistency", "sum"),
    any_anomaly_count=("any_anomaly", "sum")
).reset_index()

monthly_anomaly["anomaly_pct"] = round(monthly_anomaly["any_anomaly_count"] / monthly_anomaly["total_obs"] * 100, 2)
monthly_anomaly.to_csv("monthly_anomaly_distribution.csv", index=False)
print("Saved monthly_anomaly_distribution.csv")

print("\n=== RECENT 12 MONTHS ANOMALY RATES (Highlighting Feb-Mar 2026) ===")
print(monthly_anomaly.tail(12).to_string(index=False))

# Export sample of up to 2,000 flagged rows
flagged_rows = val_df_scan[val_df_scan["any_anomaly"]].copy()
sample_export_cols = [
    "District", "Market", "Commodity", "Variety", "Grade", "Arrival_Date",
    "Min_Price_Clean", "Modal_Price_Clean", "Max_Price_Clean", "prev_modal", "dod_ratio", "series_median",
    "flag_price_jump", "flag_non_integer", "flag_median_3x", "flag_min_max_inconsistency"
]
anomaly_sample = flagged_rows[sample_export_cols].head(2000)
anomaly_sample.to_csv("panel_anomalies_sample.csv", index=False)
print(f"Exported panel_anomalies_sample.csv ({len(anomaly_sample):,} rows)")

anomaly_summary = {
    "total_rows_scanned": total_rows,
    "flags_counts": {
        "price_jump": n_jump,
        "non_integer": n_non_int,
        "median_3x": n_med_3x,
        "min_max_inconsistency": n_min_max,
        "any_anomaly": n_any
    },
    "feb_2026_stats": monthly_anomaly[monthly_anomaly["year_month"] == "2026-02"].to_dict(orient="records"),
    "mar_2026_stats": monthly_anomaly[monthly_anomaly["year_month"] == "2026-03"].to_dict(orient="records")
}
with open("anomaly_scan_summary.json", "w") as f:
    json.dump(anomaly_summary, f, indent=2)
print("Saved anomaly_scan_summary.json")



In [ ]:
# Cell 5: Part 4 - Train-Only Eligibility (Cutoff: 2025-12-18)
train_cutoff = "2025-12-18"

# Canonical Market Normalization: strip trailing whitespace + 'APMC'
val_df["Market_Canonical"] = val_df["Market"].str.replace(r"\s+APMC$", "", regex=True)
val_df["series_id_canonical"] = (
    val_df["District"] + "__" + 
    val_df["Market_Canonical"] + "__" + 
    val_df["Commodity"] + "__" + 
    val_df["Variety"] + "__" + 
    val_df["Grade"]
)

val_train = val_df[val_df["Arrival_Date"] <= train_cutoff].copy()
val_train["Arrival_Date_dt"] = pd.to_datetime(val_train["Arrival_Date"])
print(f"Training window dataset: {len(val_train):,} rows (<= {train_cutoff})")

# 1. Profile series strictly inside training window using AS-IS series_id
train_series_metrics = []
for sid, grp in val_train.groupby("series_id"):
    dates = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n_obs = len(grp)
    first_dt = dates[0]
    last_dt = dates[-1]
    span = (last_dt - first_dt).days + 1
    cov = n_obs / span if span > 0 else 0.0
    
    train_series_metrics.append({
        "series_id": sid,
        "district": grp["District"].iloc[0],
        "commodity": grp["Commodity"].iloc[0],
        "market": grp["Market"].iloc[0],
        "variety": grp["Variety"].iloc[0],
        "grade": grp["Grade"].iloc[0],
        "n_obs_train": n_obs,
        "span_train": span,
        "cov_train": cov,
        "last_date_train": last_dt.strftime("%Y-%m-%d"),
        "recency_days_at_cutoff": (pd.Timestamp(train_cutoff) - last_dt).days
    })

df_train_series = pd.DataFrame(train_series_metrics)
df_train_series["is_tier_a"] = (df_train_series["n_obs_train"] >= 60) & (df_train_series["cov_train"] >= 0.3)
df_train_series["is_tier_b"] = (df_train_series["n_obs_train"] >= 120) & (df_train_series["cov_train"] >= 0.3)

tier_a_series = df_train_series[df_train_series["is_tier_a"]]
tier_b_series = df_train_series[df_train_series["is_tier_b"]]

print("=== TRAIN-ONLY ELIGIBILITY SUMMARY (AS-IS SERIES_ID) ===")
print("Tier A (obs>=60, cov>=0.3):")
print(f"  Series Count: {len(tier_a_series):,}")
print(f"  Unique Markets: {tier_a_series['market'].nunique():,}")
print(f"  Total Training Observations: {tier_a_series['n_obs_train'].sum():,}")

print("Tier B (obs>=120, cov>=0.3):")
print(f"  Series Count: {len(tier_b_series):,}")
print(f"  Unique Markets: {tier_b_series['market'].nunique():,}")
print(f"  Total Training Observations: {tier_b_series['n_obs_train'].sum():,}")

# 2. Profile series strictly inside training window using CANONICAL series_id_canonical
canon_series_metrics = []
for sid_canon, grp in val_train.groupby("series_id_canonical"):
    dates = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
    n_obs = len(grp)
    first_dt = dates[0]
    last_dt = dates[-1]
    span = (last_dt - first_dt).days + 1
    cov = n_obs / span if span > 0 else 0.0
    
    canon_series_metrics.append({
        "series_id_canonical": sid_canon,
        "district": grp["District"].iloc[0],
        "commodity": grp["Commodity"].iloc[0],
        "market_canonical": grp["Market_Canonical"].iloc[0],
        "variety": grp["Variety"].iloc[0],
        "grade": grp["Grade"].iloc[0],
        "n_obs_train": n_obs,
        "span_train": span,
        "cov_train": cov,
        "last_date_train": last_dt.strftime("%Y-%m-%d"),
        "recency_days_at_cutoff": (pd.Timestamp(train_cutoff) - last_dt).days
    })

df_canon_train = pd.DataFrame(canon_series_metrics)
df_canon_train["is_tier_a"] = (df_canon_train["n_obs_train"] >= 60) & (df_canon_train["cov_train"] >= 0.3)
df_canon_train["is_tier_b"] = (df_canon_train["n_obs_train"] >= 120) & (df_canon_train["cov_train"] >= 0.3)

tier_a_canon = df_canon_train[df_canon_train["is_tier_a"]]
tier_b_canon = df_canon_train[df_canon_train["is_tier_b"]]

print("\n=== TRAIN-ONLY ELIGIBILITY SUMMARY (CANONICAL MARKET NORMALIZED SERIES_ID) ===")
print("Tier A Canonical (obs>=60, cov>=0.3):")
print(f"  Series Count: {len(tier_a_canon):,}")
print(f"  Unique Markets: {tier_a_canon['market_canonical'].nunique():,}")
print(f"  Total Training Observations: {tier_a_canon['n_obs_train'].sum():,}")

print("Tier B Canonical (obs>=120, cov>=0.3):")
print(f"  Series Count: {len(tier_b_canon):,}")
print(f"  Unique Markets: {tier_b_canon['market_canonical'].nunique():,}")
print(f"  Total Training Observations: {tier_b_canon['n_obs_train'].sum():,}")

# Focus crops breakout (AS-IS)
df_train_series["is_focus"] = df_train_series["commodity"].str.contains("Wheat|Potato|Onion|Tomato|Paddy|Rice", case=False, na=False)
focus_train_breakout = []
for (dist, mkt, comm), grp in df_train_series[df_train_series["is_focus"]].groupby(["district", "market", "commodity"]):
    focus_train_breakout.append({
        "district": dist,
        "market": mkt,
        "commodity": comm,
        "total_series": len(grp),
        "tier_a_series": int(grp["is_tier_a"].sum()),
        "tier_b_series": int(grp["is_tier_b"].sum()),
        "tier_a_obs": int(grp[grp["is_tier_a"]]["n_obs_train"].sum()),
        "tier_b_obs": int(grp[grp["is_tier_b"]]["n_obs_train"].sum())
    })

df_focus_train = pd.DataFrame(focus_train_breakout)
df_focus_train.to_csv("focus_crops_train_eligibility.csv", index=False)
df_train_series.to_csv("train_eligibility_tiers.csv", index=False)
print("Saved train_eligibility_tiers.csv and focus_crops_train_eligibility.csv")

In [ ]:
# Cell 6: Parts 5 & 6 - Serving-Time Rule & Series-Level Forward Availability
test_start = "2025-12-26"

# 1. Serving-Time Rule in Training Window (Cutoff 2025-12-18)
# Under AS-IS series_id
tier_a_rec_14_asis = (tier_a_series["recency_days_at_cutoff"] <= 14).mean()
tier_a_rec_30_asis = (tier_a_series["recency_days_at_cutoff"] <= 30).mean()
tier_b_rec_14_asis = (tier_b_series["recency_days_at_cutoff"] <= 14).mean()
tier_b_rec_30_asis = (tier_b_series["recency_days_at_cutoff"] <= 30).mean()

# Under CANONICAL series_id_canonical
tier_a_rec_14_canon = (tier_a_canon["recency_days_at_cutoff"] <= 14).mean()
tier_a_rec_30_canon = (tier_a_canon["recency_days_at_cutoff"] <= 30).mean()
tier_b_rec_14_canon = (tier_b_canon["recency_days_at_cutoff"] <= 14).mean()
tier_b_rec_30_canon = (tier_b_canon["recency_days_at_cutoff"] <= 30).mean()

print("=== SERVING-TIME RULE AT TRAINING CUTOFF (2025-12-18) ===")
print(f"AS-IS Tier A Series: <=14d: {tier_a_rec_14_asis*100:.2f}% | <=30d: {tier_a_rec_30_asis*100:.2f}% (min recency: {tier_a_series['recency_days_at_cutoff'].min()}d, median: {tier_a_series['recency_days_at_cutoff'].median()}d)")
print(f"CANONICAL Tier A Series: <=14d: {tier_a_rec_14_canon*100:.2f}% | <=30d: {tier_a_rec_30_canon*100:.2f}% (min recency: {tier_a_canon['recency_days_at_cutoff'].min()}d, median: {tier_a_canon['recency_days_at_cutoff'].median()}d)")

# 2. Test-Window Serving Viability across all test observations (Arrival_Date >= 2025-12-26)
val_full_sorted = val_df.sort_values(by=["series_id_canonical", "Arrival_Date"]).copy()
val_full_sorted["Arrival_Date_dt"] = pd.to_datetime(val_full_sorted["Arrival_Date"])
val_full_sorted["prev_obs_date"] = val_full_sorted.groupby("series_id_canonical")["Arrival_Date_dt"].shift(1)
val_full_sorted["gap_to_prev_obs"] = (val_full_sorted["Arrival_Date_dt"] - val_full_sorted["prev_obs_date"]).dt.days

# Also compute for as-is series
val_full_asis = val_df.sort_values(by=["series_id", "Arrival_Date"]).copy()
val_full_asis["Arrival_Date_dt"] = pd.to_datetime(val_full_asis["Arrival_Date"])
val_full_asis["prev_obs_date"] = val_full_asis.groupby("series_id")["Arrival_Date_dt"].shift(1)
val_full_asis["gap_to_prev_obs"] = (val_full_asis["Arrival_Date_dt"] - val_full_asis["prev_obs_date"]).dt.days

tier_a_canon_ids = set(tier_a_canon["series_id_canonical"])
tier_b_canon_ids = set(tier_b_canon["series_id_canonical"])
tier_a_asis_ids = set(tier_a_series["series_id"])
tier_b_asis_ids = set(tier_b_series["series_id"])

test_canon = val_full_sorted[val_full_sorted["Arrival_Date"] >= test_start]
test_asis = val_full_asis[val_full_asis["Arrival_Date"] >= test_start]

test_a_canon = test_canon[test_canon["series_id_canonical"].isin(tier_a_canon_ids)]
test_b_canon = test_canon[test_canon["series_id_canonical"].isin(tier_b_canon_ids)]
test_a_asis = test_asis[test_asis["series_id"].isin(tier_a_asis_ids)]
test_b_asis = test_asis[test_asis["series_id"].isin(tier_b_asis_ids)]

canon_test_a_pass14 = float((test_a_canon["gap_to_prev_obs"] <= 14).mean()) if len(test_a_canon) > 0 else 0.0
canon_test_b_pass14 = float((test_b_canon["gap_to_prev_obs"] <= 14).mean()) if len(test_b_canon) > 0 else 0.0
asis_test_a_pass14 = float((test_a_asis["gap_to_prev_obs"] <= 14).mean()) if len(test_a_asis) > 0 else 0.0
asis_test_b_pass14 = float((test_b_asis["gap_to_prev_obs"] <= 14).mean()) if len(test_b_asis) > 0 else 0.0

print(f"\n=== TEST-WINDOW SERVING VIABILITY (Test Origins >= {test_start}, Gap<=14d) ===")
print(f"AS-IS Tier A Test Origins: {len(test_a_asis):,} | Pass 14d: {asis_test_a_pass14*100:.2f}% (0 because 2026 panel has APMC suffix)")
print(f"CANONICAL Tier A Test Origins: {len(test_a_canon):,} | Pass 14d: {canon_test_a_pass14*100:.2f}%")
print(f"CANONICAL Tier B Test Origins: {len(test_b_canon):,} | Pass 14d: {canon_test_b_pass14*100:.2f}%")

# 3. Series-Level (Unweighted) Forward Availability
def get_series_level_availability(series_subset_df, id_col):
    t7_shares = []
    t6_8_shares = []
    for sid, grp in series_subset_df.groupby(id_col):
        dates = sorted(grp["Arrival_Date_dt"].drop_duplicates().tolist())
        date_set = set(dates)
        has_t7_list = []
        has_t6_8_list = []
        for t in dates:
            has_t7 = (t + datetime.timedelta(days=7)) in date_set
            has_t6_8 = any((t + datetime.timedelta(days=k)) in date_set for k in [6, 7, 8])
            has_t7_list.append(has_t7)
            has_t6_8_list.append(has_t6_8)
        t7_shares.append(float(np.mean(has_t7_list)))
        t6_8_shares.append(float(np.mean(has_t6_8_list)))
    return np.array(t7_shares), np.array(t6_8_shares)

val_tier_a_canon = val_full_sorted[val_full_sorted["series_id_canonical"].isin(tier_a_canon_ids)]
val_tier_b_canon = val_full_sorted[val_full_sorted["series_id_canonical"].isin(tier_b_canon_ids)]
t7_a, t6_8_a = get_series_level_availability(val_tier_a_canon, "series_id_canonical")
t7_b, t6_8_b = get_series_level_availability(val_tier_b_canon, "series_id_canonical")

def calc_dist(arr):
    return {
        "count": len(arr),
        "min": round(float(np.min(arr)), 4),
        "p10": round(float(np.percentile(arr, 10)), 4),
        "p25": round(float(np.percentile(arr, 25)), 4),
        "median": round(float(np.median(arr)), 4),
        "p75": round(float(np.percentile(arr, 75)), 4),
        "p90": round(float(np.percentile(arr, 90)), 4),
        "max": round(float(np.max(arr)), 4),
        "mean": round(float(np.mean(arr)), 4)
    }

series_avail_summary = {
    "tier_a_canonical": {
        "series_count": len(t7_a),
        "t7_distribution": calc_dist(t7_a),
        "t6_8_distribution": calc_dist(t6_8_a)
    },
    "tier_b_canonical": {
        "series_count": len(t7_b),
        "t7_distribution": calc_dist(t7_b),
        "t6_8_distribution": calc_dist(t6_8_b)
    }
}

serving_summary = {
    "training_cutoff": train_cutoff,
    "test_start_date": test_start,
    "as_is_series_id": {
        "tier_a_recency_14d_share": round(float(tier_a_rec_14_asis), 4),
        "tier_a_recency_30d_share": round(float(tier_a_rec_30_asis), 4),
        "tier_b_recency_14d_share": round(float(tier_b_rec_14_asis), 4),
        "tier_b_recency_30d_share": round(float(tier_b_rec_30_asis), 4),
        "tier_a_test_origins_count": len(test_a_asis),
        "tier_a_test_origins_pass_14d_share": round(float(asis_test_a_pass14), 4),
        "tier_b_test_origins_count": len(test_b_asis),
        "tier_b_test_origins_pass_14d_share": round(float(asis_test_b_pass14), 4),
        "explanation": "As-is series_id includes 'Market'. In late 2025/2026, raw data appended ' APMC' to market names, causing all series to fork. As a result, non-APMC series lack 2026 observations and APMC series had <60 training rows."
    },
    "canonical_market_normalized": {
        "tier_a_recency_14d_share": round(float(tier_a_rec_14_canon), 4),
        "tier_a_recency_30d_share": round(float(tier_a_rec_30_canon), 4),
        "tier_b_recency_14d_share": round(float(tier_b_rec_14_canon), 4),
        "tier_b_recency_30d_share": round(float(tier_b_rec_30_canon), 4),
        "tier_a_test_origins_count": len(test_a_canon),
        "tier_a_test_origins_pass_14d_share": round(float(canon_test_a_pass14), 4),
        "tier_b_test_origins_count": len(test_b_canon),
        "tier_b_test_origins_pass_14d_share": round(float(canon_test_b_pass14), 4)
    }
}

with open("serving_rule_feasibility.json", "w") as f:
    json.dump(serving_summary, f, indent=2)

with open("series_level_availability_summary.json", "w") as f:
    json.dump(series_avail_summary, f, indent=2)

print("Saved serving_rule_feasibility.json and series_level_availability_summary.json")

In [ ]:
# Cell 7: Part 7 - Rolling-Origin Cross-Validation Feasibility (4 Expanding Folds)
end_date = pd.Timestamp("2026-09-30")
cur_end = end_date
folds_def = []

for fold_num in [4, 3, 2, 1]:
    t_end = cur_end
    t_start = t_end - pd.Timedelta(days=59) # 60 days inclusive
    emb_end = t_start - pd.Timedelta(days=1)
    emb_start = t_start - pd.Timedelta(days=7) # 7 dates embargo
    tr_end = emb_start - pd.Timedelta(days=1)
    
    # Check overlap with 2026-02-22..2026-03-31
    disc_start = pd.Timestamp("2026-02-22")
    disc_end = pd.Timestamp("2026-03-31")
    overlaps_disc_test = bool((t_start <= disc_end) and (t_end >= disc_start))
    overlaps_disc_train = bool((tr_end >= disc_start))
    
    folds_def.append({
        "fold": fold_num,
        "train_start": "2021-09-01",
        "train_end": tr_end.strftime("%Y-%m-%d"),
        "embargo_start": emb_start.strftime("%Y-%m-%d"),
        "embargo_end": emb_end.strftime("%Y-%m-%d"),
        "embargo_dates_count": 7,
        "test_start": t_start.strftime("%Y-%m-%d"),
        "test_end": t_end.strftime("%Y-%m-%d"),
        "test_days_count": 60,
        "test_overlaps_feb_mar_discrepancy": overlaps_disc_test,
        "train_overlaps_feb_mar_discrepancy": overlaps_disc_train
    })
    cur_end = t_start - pd.Timedelta(days=1)

folds_def = sorted(folds_def, key=lambda x: x["fold"])

rolling_results = []
for f in folds_def:
    t_s = f["test_start"]
    t_e = f["test_end"]
    
    # 1. AS-IS series
    fold_test_asis = val_full_asis[(val_full_asis["Arrival_Date"] >= t_s) & (val_full_asis["Arrival_Date"] <= t_e)]
    asis_counts = fold_test_asis.groupby("series_id").size().to_dict()
    tier_a_asis_counts = [asis_counts.get(sid, 0) for sid in tier_a_asis_ids]
    tier_b_asis_counts = [asis_counts.get(sid, 0) for sid in tier_b_asis_ids]
    tier_a_asis_ge_20 = sum(1 for c in tier_a_asis_counts if c >= 20)
    tier_b_asis_ge_20 = sum(1 for c in tier_b_asis_counts if c >= 20)
    
    # 2. CANONICAL series
    fold_test_canon = val_full_sorted[(val_full_sorted["Arrival_Date"] >= t_s) & (val_full_sorted["Arrival_Date"] <= t_e)]
    canon_counts = fold_test_canon.groupby("series_id_canonical").size().to_dict()
    tier_a_canon_counts = [canon_counts.get(sid, 0) for sid in tier_a_canon_ids]
    tier_b_canon_counts = [canon_counts.get(sid, 0) for sid in tier_b_canon_ids]
    tier_a_canon_ge_20 = sum(1 for c in tier_a_canon_counts if c >= 20)
    tier_b_canon_ge_20 = sum(1 for c in tier_b_canon_counts if c >= 20)
    
    rolling_results.append({
        "fold": f["fold"],
        "train_window": f"{f['train_start']} to {f['train_end']}",
        "embargo_window": f"{f['embargo_start']} to {f['embargo_end']}",
        "test_window": f"{f['test_start']} to {f['test_end']}",
        "test_overlaps_feb_mar_discrepancy": f["test_overlaps_feb_mar_discrepancy"],
        "train_overlaps_feb_mar_discrepancy": f["train_overlaps_feb_mar_discrepancy"],
        "asis_tier_a_total_series": len(tier_a_asis_ids),
        "asis_tier_a_series_ge_20_origins": tier_a_asis_ge_20,
        "asis_tier_a_ge_20_pct": round(tier_a_asis_ge_20 / len(tier_a_asis_ids) * 100, 2),
        "canon_tier_a_total_series": len(tier_a_canon_ids),
        "canon_tier_a_series_ge_20_origins": tier_a_canon_ge_20,
        "canon_tier_a_ge_20_pct": round(tier_a_canon_ge_20 / len(tier_a_canon_ids) * 100, 2),
        "canon_tier_b_total_series": len(tier_b_canon_ids),
        "canon_tier_b_series_ge_20_origins": tier_b_canon_ge_20,
        "canon_tier_b_ge_20_pct": round(tier_b_canon_ge_20 / len(tier_b_canon_ids) * 100, 2)
    })

print("=== ROLLING-ORIGIN FEASIBILITY SUMMARY (4 Folds) ===")
df_rolling = pd.DataFrame(rolling_results)
print(df_rolling.to_string(index=False))

with open("rolling_origin_feasibility.json", "w") as f:
    json.dump({"folds": rolling_results}, f, indent=2)
print("Saved rolling_origin_feasibility.json")

In [ ]:
# Cell 8: Part 8 - Final Output Artifact Verification & Hashes
print("=== GENERATED ARTIFACTS VERIFICATION ===")
expected_artifacts = [
    "diff_77_deep_analysis.csv",
    "diff_77_deep_analysis.json",
    "feb_mar_cluster_weekly.csv",
    "feb_mar_cluster_summary.json",
    "monthly_anomaly_distribution.csv",
    "panel_anomalies_sample.csv",
    "anomaly_scan_summary.json",
    "train_eligibility_tiers.csv",
    "focus_crops_train_eligibility.csv",
    "serving_rule_feasibility.json",
    "series_level_availability_summary.json",
    "rolling_origin_feasibility.json"
]

manifest_outputs = []
for fname in expected_artifacts:
    if os.path.exists(fname):
        sz = os.path.getsize(fname)
        sha = compute_sha256(fname)
        line_cnt = None
        if fname.endswith(".csv"):
            with open(fname) as f:
                line_cnt = sum(1 for _ in f)
        manifest_outputs.append({
            "artifact": fname,
            "size_bytes": sz,
            "sha256": sha,
            "line_count": line_cnt
        })
        print(f"FILE: {fname:<36} | {sz:>10,} bytes | lines: {str(line_cnt):<8} | sha256: {sha}")
    else:
        print(f"MISSING: {fname}")

with open("kernel_execution_manifest.json", "w") as f:
    json.dump({"artifacts": manifest_outputs, "timestamp_utc": time.time()}, f, indent=2)
print("\nKernel Execution Completed Successfully!")

